# 02 · Second-Order ODEs: The Damped Harmonic Oscillator

**Level:** Beginner → Intermediate bridge
**Prerequisites:** `01_intro_to_pinn`

In notebook 01 we solved a first-order ODE, `dy/dx = -y`. Real engineering systems —
suspension systems, RLC circuits, molecular bonds, control loops — are usually
governed by **second-order** dynamics. This notebook shows how a PINN handles the
extra derivative, how to impose *two* initial conditions, and what "stiffness"
(high frequency) does to training.

### What you will learn
1. Formulating a second-order ODE residual with `autograd` (nested differentiation)
2. Imposing initial conditions on both the value **and** the first derivative
3. Comparing the PINN against the exact analytical solution
4. Seeing why higher oscillation frequency makes PINNs harder to train

### Associated references
- Raissi, Perdikaris & Karniadakis (2019), *J. Comput. Phys.* — foundational PINN paper
- Wang, Sankaran & Perdikaris (2022), *Respecting causality is all you need* — why temporal/oscillatory problems are hard


## 1 · The physics

A mass on a spring with damping obeys

$$m\,\frac{d^2u}{dt^2} + c\,\frac{du}{dt} + k\,u = 0.$$

Dividing by $m$ and writing $2\zeta\omega_0 = c/m$, $\omega_0^2 = k/m$:

$$\frac{d^2u}{dt^2} + 2\zeta\omega_0\,\frac{du}{dt} + \omega_0^2 u = 0,
\qquad u(0)=u_0,\quad \frac{du}{dt}(0)=v_0.$$

For the **under-damped** case ($\zeta < 1$) the exact solution is

$$u(t) = e^{-\zeta\omega_0 t}\left[u_0\cos(\omega_d t) + \frac{v_0 + \zeta\omega_0 u_0}{\omega_d}\sin(\omega_d t)\right],
\qquad \omega_d = \omega_0\sqrt{1-\zeta^2}.$$

We will train a PINN to reproduce this using **only** the equation and the two
initial conditions — no solution data.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- Physical parameters ---
omega0 = 3.0     # natural frequency (rad/s)
zeta   = 0.20    # damping ratio (under-damped)
u0, v0 = 1.0, 0.0
T      = 6.0     # time horizon

omega_d = omega0 * np.sqrt(1 - zeta**2)

def exact(t):
    t = np.asarray(t)
    env = np.exp(-zeta * omega0 * t)
    return env * (u0*np.cos(omega_d*t) + (v0 + zeta*omega0*u0)/omega_d*np.sin(omega_d*t))


In [ ]:
# --- Network ---
class MLP(nn.Module):
    def __init__(self, layers):
        super().__init__()
        seq = []
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2:
                seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, t):
        return self.net(t)

model = MLP([1, 64, 64, 64, 1]).to(device)
print(model)


In [ ]:
# --- Collocation points (where we enforce the ODE) ---
N_col = 400
t_col = torch.rand(N_col, 1, device=device) * T
t_col.requires_grad_(True)

# Initial-condition point
t0 = torch.zeros(1, 1, device=device, requires_grad=True)

def residual(model, t):
    u   = model(t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_tt= torch.autograd.grad(u_t, t, torch.ones_like(u_t), create_graph=True)[0]
    return u_tt + 2*zeta*omega0*u_t + omega0**2 * u


In [ ]:
# --- Loss: physics residual + two initial conditions ---
def loss_fn(model):
    r = residual(model, t_col)
    loss_phys = torch.mean(r**2)

    u_pred  = model(t0)
    u_t0    = torch.autograd.grad(u_pred, t0, torch.ones_like(u_pred), create_graph=True)[0]
    loss_ic = (u_pred - u0)**2 + (u_t0 - v0)**2      # value AND velocity

    # weight IC more strongly so it is respected early
    return loss_phys + 20.0*loss_ic.squeeze(), loss_phys.item(), loss_ic.item()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
EPOCHS = 8000
hist = []
for ep in range(EPOCHS):
    optimizer.zero_grad()
    loss, lp, li = loss_fn(model)
    loss.backward(); optimizer.step()
    hist.append(loss.item())
    if ep % 1000 == 0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e} | phys {lp:.3e} | ic {li:.3e}")
print("done")


In [ ]:
# --- Evaluate & plot ---
tt = torch.linspace(0, T, 400, device=device).view(-1,1)
with torch.no_grad():
    up = model(tt).cpu().numpy().ravel()
tt_np = tt.cpu().numpy().ravel()
ue = exact(tt_np)
l2 = np.linalg.norm(up-ue)/np.linalg.norm(ue)
print(f"relative L2 error: {l2:.3e}")

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(tt_np, ue, 'k-', lw=2, label='exact')
ax[0].plot(tt_np, up, 'r--', lw=2, label='PINN')
ax[0].set_xlabel('t'); ax[0].set_ylabel('u(t)'); ax[0].legend()
ax[0].set_title(f'Damped oscillator (rel L2 = {l2:.2e})')
ax[1].semilogy(hist); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('loss')
ax[1].set_title('Training loss'); plt.tight_layout(); plt.show()


## 2 · Why the initial-derivative condition matters

A second-order ODE needs **two** pieces of information to pin down a unique
solution. If we only imposed $u(0)=u_0$, infinitely many trajectories with
different starting velocities would satisfy the equation. That is why the loss
contains both $(u(0)-u_0)^2$ and $(u'(0)-v_0)^2$.

Notice we compute $u'(0)$ with the **same** `autograd` machinery used for the
residual — the derivative of the network is a first-class citizen we can put
anywhere in the loss.


In [ ]:
# --- Hard-constraint variant: IC satisfied exactly by construction ---
# u_ansatz(t) = u0 + v0*t + t^2 * N(t)  ->  u(0)=u0, u'(0)=v0 automatically
class HardIC(nn.Module):
    def __init__(self, layers, u0, v0):
        super().__init__(); self.core = MLP(layers); self.u0=u0; self.v0=v0
    def forward(self, t):
        return self.u0 + self.v0*t + (t**2) * self.core(t)

model2 = HardIC([1,64,64,64,1], u0, v0).to(device)
opt2 = torch.optim.Adam(model2.parameters(), lr=2e-3)
for ep in range(6000):
    opt2.zero_grad()
    r = residual(model2, t_col)         # only the physics loss now!
    loss = torch.mean(r**2)
    loss.backward(); opt2.step()
with torch.no_grad():
    up2 = model2(tt).cpu().numpy().ravel()
l2_2 = np.linalg.norm(up2-ue)/np.linalg.norm(ue)
print(f"hard-constraint rel L2 error: {l2_2:.3e}  (vs soft {l2:.3e})")


## Summary

- Second-order ODEs need **nested** `autograd` calls and **two** initial conditions.
- Weighting or hard-constraining the initial conditions dramatically improves training.
- Higher frequencies expose the **spectral bias** of PINNs — the recurring theme
  that motivates most of the advanced techniques in this series.

**Next:** `02_Intermediate/03_heat_equation` — our first genuine PDE with a
spatial *and* temporal dimension.
